In [1]:
!pip install pyspark py4j

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425345 sha256=fbb7d29ebfebd300ad7e41695fa67f11f2bead684e8c649099f41e2f35f3a28e
  Stored in directory: /root/.cache/pip/wheels/41/4e/10/c2cf2467f71c678cfc8a6b9ac9241e5e44a01940da8fbb17fc
Successfully built pyspark


In [2]:
import os
import sys
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql import types as T
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
plt.style.use(style='seaborn')

from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler

<ipython-input-2-a6af86bdc90a>:9: MatplotlibDeprecationWarning: The seaborn styles shipped by Matplotlib are deprecated since 3.6, as they no longer correspond to the styles shipped by seaborn. However, they will remain available as 'seaborn-v0_8-<style>'. Alternatively, directly use the seaborn API instead.
  plt.style.use(style='seaborn')


In [3]:
spark = SparkSession.builder.appName("mySparkSession").getOrCreate()

In [4]:
# mount to Google drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
# Spark dataframe
df = spark.read.csv('/content/drive/MyDrive/mySharedDrive/CSCI316/Datasets/trainEditedSJv3(500).csv', header = True, inferSchema = True)

In [6]:
# Pandas dataframe
pdDf = pd.read_csv('/content/drive/MyDrive/mySharedDrive/CSCI316/Datasets/trainEditedSJv3(500).csv')

In [7]:
# Print the shape of the Spark dataframe
print((df.count(), len(df.columns)))

(499, 82)


In [8]:
# Show the content of the Spark dataframe
df.show(truncate = False)

+------------------+----------------+--------------------+-----------------+---------------------+-------------------+-----------------------+-----------------+---------------------+---------------+-------------------+-----------+------------+-----------+-------------+-----------+---------------+---------+-------------+-----------+-----------+------------------+----------------------+-------------------+-----------------------+---------------------+-------------------------+-------------------+-----------------------+-----------------+---------------------+------------+----------------+-------------+-----------------+---------------+-------------------+-------------+-----------------+-----------+---------------+---------------------+-------------------------+----------------------+--------------------------+------------------------+----------------------------+----------------------+--------------------------+--------------------+------------------------+---------------+---------------

In [9]:
# Print the schema of the Spark dataframe
df.printSchema()

root
 |-- number_of_elements: integer (nullable = true)
 |-- mean_atomic_mass: double (nullable = true)
 |-- wtd_mean_atomic_mass: double (nullable = true)
 |-- gmean_atomic_mass: double (nullable = true)
 |-- wtd_gmean_atomic_mass: double (nullable = true)
 |-- entropy_atomic_mass: double (nullable = true)
 |-- wtd_entropy_atomic_mass: double (nullable = true)
 |-- range_atomic_mass: double (nullable = true)
 |-- wtd_range_atomic_mass: double (nullable = true)
 |-- std_atomic_mass: double (nullable = true)
 |-- wtd_std_atomic_mass: double (nullable = true)
 |-- mean_fie: double (nullable = true)
 |-- wtd_mean_fie: double (nullable = true)
 |-- gmean_fie: double (nullable = true)
 |-- wtd_gmean_fie: double (nullable = true)
 |-- entropy_fie: double (nullable = true)
 |-- wtd_entropy_fie: double (nullable = true)
 |-- range_fie: double (nullable = true)
 |-- wtd_range_fie: double (nullable = true)
 |-- std_fie: double (nullable = true)
 |-- wtd_std_fie: double (nullable = true)
 |-- mea

In [10]:
# Show the Pandas dataframe
pdDf.head()

,number_of_elements,mean_atomic_mass,wtd_mean_atomic_mass,gmean_atomic_mass,wtd_gmean_atomic_mass,entropy_atomic_mass,wtd_entropy_atomic_mass,range_atomic_mass,wtd_range_atomic_mass,std_atomic_mass,...,wtd_mean_Valence,gmean_Valence,wtd_gmean_Valence,entropy_Valence,wtd_entropy_Valence,range_Valence,wtd_range_Valence,std_Valence,wtd_std_Valence,critical_temp
0,2,155.892284,195.159301,150.383867,194.641796,0.658023,0.069126,82.148569,190.107308,41.074284,...,4.956000,3.872983,4.944124,0.661563,0.070741,2,4.824000,1.000000,0.293367,0
1,4,76.517718,57.541467,59.310096,36.009693,1.197273,0.899625,122.906070,38.069238,44.289459,...,2.278571,2.213364,2.239154,1.368922,1.006120,1,1.128571,0.433013,0.448296,0
2,5,103.563810,60.564038,79.403743,36.787423,1.434668,1.449081,141.250600,17.215371,54.555995,...,2.442857,2.861938,2.349498,1.564132,1.349598,2,1.057143,0.894427,0.749013,0
3,5,72.952854,56.781088,59.186241,35.757198,1.445795,1.006723,122.906070,36.375341,40.250192,...,2.264286,2.168944,2.226222,1.594167,1.070026,1,1.131429,0.400000,0.440952,0
4,5,72.952854,57.314739,59.186241,35.934573,1.445795,0.935036,122.906070,38.015938,40.250192,...,2.274286,2.168944,2.235267,1.594167,1.027500,1,1.140000,0.400000,0.446154,0


In [11]:
# Pandas dataframe information
pdDf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499 entries, 0 to 498
Data columns (total 82 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   number_of_elements               499 non-null    int64  
 1   mean_atomic_mass                 499 non-null    float64
 2   wtd_mean_atomic_mass             499 non-null    float64
 3   gmean_atomic_mass                499 non-null    float64
 4   wtd_gmean_atomic_mass            499 non-null    float64
 5   entropy_atomic_mass              499 non-null    float64
 6   wtd_entropy_atomic_mass          499 non-null    float64
 7   range_atomic_mass                499 non-null    float64
 8   wtd_range_atomic_mass            499 non-null    float64
 9   std_atomic_mass                  499 non-null    float64
 10  wtd_std_atomic_mass              499 non-null    float64
 11  mean_fie                         499 non-null    float64
 12  wtd_mean_fie          

# Preprocessing
# The following examples are based on Spark dataframe

In [12]:
# Print the name of column headers
df.columns

['number_of_elements',
 'mean_atomic_mass',
 'wtd_mean_atomic_mass',
 'gmean_atomic_mass',
 'wtd_gmean_atomic_mass',
 'entropy_atomic_mass',
 'wtd_entropy_atomic_mass',
 'range_atomic_mass',
 'wtd_range_atomic_mass',
 'std_atomic_mass',
 'wtd_std_atomic_mass',
 'mean_fie',
 'wtd_mean_fie',
 'gmean_fie',
 'wtd_gmean_fie',
 'entropy_fie',
 'wtd_entropy_fie',
 'range_fie',
 'wtd_range_fie',
 'std_fie',
 'wtd_std_fie',
 'mean_atomic_radius',
 'wtd_mean_atomic_radius',
 'gmean_atomic_radius',
 'wtd_gmean_atomic_radius',
 'entropy_atomic_radius',
 'wtd_entropy_atomic_radius',
 'range_atomic_radius',
 'wtd_range_atomic_radius',
 'std_atomic_radius',
 'wtd_std_atomic_radius',
 'mean_Density',
 'wtd_mean_Density',
 'gmean_Density',
 'wtd_gmean_Density',
 'entropy_Density',
 'wtd_entropy_Density',
 'range_Density',
 'wtd_range_Density',
 'std_Density',
 'wtd_std_Density',
 'mean_ElectronAffinity',
 'wtd_mean_ElectronAffinity',
 'gmean_ElectronAffinity',
 'wtd_gmean_ElectronAffinity',
 'entropy_E

In [13]:
# Shorten the feature's name
from functools import reduce

oldColumns = df.schema.names

# Specify the new column name
newColumns = ['NumOfElem',
 'MAtomM',
 'WMAtomM',
 'GMAtomM',
 'WGMAtomM',
 'EAtomM',
 'WEAtomM',
 'RAtomM',
 'WRAtomM',
 'StdAtomM',
 'WStdAtomM',
 'MFie',
 'WMFie',
 'GMFie',
 'WGMFie',
 'EFie',
 'WEFie',
 'RFie',
 'WRFie',
 'StdFie',
 'WStdFie',
 'MAtomRad',
 'WMAtomRad',
 'GMAtomRad',
 'WGMAtomRad',
 'EAtomRad',
 'WEAtomRad',
 'RAtomRad',
 'WRAtomRad',
 'StdAtomRad',
 'WStdAtomRad',
 'MDensity',
 'WMDensity',
 'GMDensity',
 'WGMDensity',
 'EDensity',
 'WEDensity',
 'RDensity',
 'WRDensity',
 'StdDensity',
 'WStdDensity',
 'MElecAffinity',
 'WMElecAffinity',
 'GMElecAffinity',
 'WGMElecAffinity',
 'EElecAffinity',
 'WEElecAffinity',
 'RElecAffinity',
 'WRElecAffinity',
 'StdElecAffinity',
 'WStdElecAffinity',
 'MFusHeat',
 'WmFusHeat',
 'GMFusHeat',
 'WGMFusHeat',
 'EFusHeat',
 'WEFusHeat',
 'RFusHeat',
 'WRFusHeat',
 'StdFusHeat',
 'WStdFusHeat',
 'MThermalCon',
 'WMThermalCon',
 'GMThermalCon',
 'WGMThermalCon',
 'EThermalCon',
 'WEThermalCon',
 'RThermalCon',
 'WRThermalCon',
 'StdThermalCon',
 'WStdThermalCon',
 'MValence',
 'WMValence',
 'GMValence',
 'WGMValence',
 'EValence',
 'WEValence',
 'RValence',
 'WRValence',
 'StdValence',
 'WStdValence',
 'CriticalTemp']

# Transform the old column name to new column name
df = reduce(lambda df, idx: df.withColumnRenamed(oldColumns[idx], newColumns[idx]),range(len(oldColumns)), df)

# print the schema
df.printSchema()

root
 |-- NumOfElem: integer (nullable = true)
 |-- MAtomM: double (nullable = true)
 |-- WMAtomM: double (nullable = true)
 |-- GMAtomM: double (nullable = true)
 |-- WGMAtomM: double (nullable = true)
 |-- EAtomM: double (nullable = true)
 |-- WEAtomM: double (nullable = true)
 |-- RAtomM: double (nullable = true)
 |-- WRAtomM: double (nullable = true)
 |-- StdAtomM: double (nullable = true)
 |-- WStdAtomM: double (nullable = true)
 |-- MFie: double (nullable = true)
 |-- WMFie: double (nullable = true)
 |-- GMFie: double (nullable = true)
 |-- WGMFie: double (nullable = true)
 |-- EFie: double (nullable = true)
 |-- WEFie: double (nullable = true)
 |-- RFie: double (nullable = true)
 |-- WRFie: double (nullable = true)
 |-- StdFie: double (nullable = true)
 |-- WStdFie: double (nullable = true)
 |-- MAtomRad: double (nullable = true)
 |-- WMAtomRad: double (nullable = true)
 |-- GMAtomRad: double (nullable = true)
 |-- WGMAtomRad: double (nullable = true)
 |-- EAtomRad: double (null

# Check for NULL values

In [14]:
# Check if there is any NULL values. If yes, remove them
from pyspark.sql.functions import col, count, isnan, when
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+---------+------+-------+-------+--------+------+-------+------+-------+--------+---------+----+-----+-----+------+----+-----+----+-----+------+-------+--------+---------+---------+----------+--------+---------+--------+---------+----------+-----------+--------+---------+---------+----------+--------+---------+--------+---------+----------+-----------+-------------+--------------+--------------+---------------+-------------+--------------+-------------+--------------+---------------+----------------+--------+---------+---------+----------+--------+---------+--------+---------+----------+-----------+-----------+------------+------------+-------------+-----------+------------+-----------+------------+-------------+--------------+--------+---------+---------+----------+--------+---------+--------+---------+----------+-----------+------------+
|NumOfElem|MAtomM|WMAtomM|GMAtomM|WGMAtomM|EAtomM|WEAtomM|RAtomM|WRAtomM|StdAtomM|WStdAtomM|MFie|WMFie|GMFie|WGMFie|EFie|WEFie|RFie|WRFie|StdFie|WS

In [ ]:
#pdDf = df.toPandas()
#g = sns.pairplot(pdDf, diag_kind='kde')

#### The result indicates there is no null value features

In [ ]:
# split the dataset into 67% training and 33% test
(train, test) = df.randomSplit([0.67,0.33])

train.show(10, truncate = False)
test.show(10, truncate = False)

In [ ]:
# Print the dataset
print(f'Train data set size: {train.count()} records')
print(f'Test data set size: {test.count()} records')

In [ ]:
train.printSchema()
test.printSchema()

In [ ]:
# Checking data type
catCols = [x for (x, dataType) in train.dtypes if dataType == 'string']
numCols = [x for (x, dataType) in train.dtypes if ((dataType == 'double') & (x != 'isFound'))]

print(catCols)
print(numCols)

In [ ]:
# split the features and label. We want to determine the critical temperature, hence, CriticalTemp is the label.
# exclude it from other features.
features = df.drop('CriticalTemp')

In [ ]:
# Preparing the dataset for regression model
# Assemble all features together using VectorAssembler
myAssembler = VectorAssembler(
    inputCols = features.columns,
    outputCol = 'features'
)

output = myAssembler.transform(df).select('features','CriticalTemp')
output.show(truncate = False)

# Splitting the dataset into 67% training and 33% test

In [ ]:
# split the dataset into 67% training and 33% test
(train, test) = output.randomSplit([0.67,0.33])

train.show(10, truncate = False)
test.show(10, truncate = False)

#### Linear Regression

In [ ]:
# import the LinearRegression model
from pyspark.ml.regression import LinearRegression

# Create an instance of LinearRegression model
linReg = LinearRegression(featuresCol = 'features', labelCol = 'CriticalTemp')

# Train the model using train dataset
linearModel = linReg.fit(train)

In [ ]:
# Print the coefficients and Intercept
print('Coefficients: ' + str(linearModel.coefficients))
print('\nIntercept: ' + str(linearModel.intercept))

In [ ]:
# Print the training summary
trainSummary = linearModel.summary
print('RMSE: %f' % trainSummary.rootMeanSquaredError)
print('\nr2: %f' % trainSummary.r2)

In [ ]:
from pyspark.sql.functions import abs
# predictions = linearModel.transform(test)
# x = ((predictions['YearlyAmtSpent']-predictions['prediction'])/predictions['YearlyAmtSpent']) * 100
# predictions = predictions.withColumn('Accuracy', abs(x))
# predictions.select('prediction', 'YearlyAmtSpent', 'Accuracy', 'features').show(truncate = False)
predictions = linearModel.transform(test)
x = ((predictions['CriticalTemp']-predictions['prediction'])/predictions['CriticalTemp']) * 100
predictions = predictions.withColumn('Accuracy', abs(x))
predictions.select('prediction', 'CriticalTemp', 'Accuracy', 'features').show(truncate = False)

In [ ]:
# Import the RegressionEvaluator to evaluate the predicted result
from pyspark.ml.evaluation import RegressionEvaluator
myPredEvaluator = RegressionEvaluator (
    predictionCol = 'prediction',
    labelCol = 'CriticalTemp',
    metricName = 'r2'
)

print('R Squared (R2) on test data = %g' % myPredEvaluator.evaluate(predictions))

In [ ]:
# Define an adjustment function to adjust a dataset
def adjR2(x):
    r2 = trainSummary.r2
    n = df.count()
    p = len(df.columns)
    adjustedR2 = 1 - (1 - r2) * (n - 1) / (n-p-1)
    return adjustedR2

In [ ]:
# Call the function adjR2 to adjust the training dataset
adjR2(train)

In [ ]:
# Call the function adjR2 to adjust the test dataset
adjR2(test)

In [ ]:
linReg = LinearRegression(
    featuresCol = 'features',
    labelCol = 'CriticalTemp',
    maxIter = 50,
    regParam = 0.12,
    elasticNetParam = 0.2)
linearModel = linReg.fit(train)

In [ ]:
linearModel.summary.rootMeanSquaredError

In [ ]:
featuresRDD = features.rdd.map(lambda row: row[0:])

In [ ]:
featuresRDD.collect()

In [ ]:
from pyspark.mllib.feature import StandardScaler
scaler1 = StandardScaler().fit(featuresRDD)
scaledFeatures = scaler1.transform(featuresRDD)

In [ ]:
for data in scaledFeatures.collect():
    print(data)

In [ ]:
df2 = scaledFeatures.map(lambda x: (x, )).toDF(['ScaledData'])

In [ ]:
df2.show(truncate = False)

#### Random Forest Regressor

In [ ]:
# Random Forest
from pyspark.ml.regression import RandomForestRegressor
# randomForestReg = RandomForestRegressor(featuresCol = 'features', labelCol = 'YearlyAmtSpent')
# randomForestModel = randomForestReg.fit(train)
randomForestReg = RandomForestRegressor(featuresCol = 'features', labelCol = 'CriticalTemp')
randomForestModel = randomForestReg.fit(train)

In [ ]:
predictions = randomForestModel.transform(test)
predictions.show(truncate = False)

In [ ]:
from pyspark.sql.functions import abs
# predictions = randomForestModel.transform(test)
# x = ((predictions['YearlyAmtSpent']-predictions['prediction'])/predictions['YearlyAmtSpent']) * 100
# predictions = predictions.withColumn('Accuracy', abs(x))
# predictions.select('prediction', 'YearlyAmtSpent', 'Accuracy', 'features').show(truncate = False)
predictions = randomForestModel.transform(test)
x = ((predictions['CriticalTemp']-predictions['prediction'])/predictions['CriticalTemp']) * 100
predictions = predictions.withColumn('Accuracy', abs(x))
predictions.select('prediction', 'CriticalTemp', 'Accuracy', 'features').show(truncate = False)

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
myPredEvaluator = RegressionEvaluator (
    predictionCol = 'prediction',
    labelCol = 'CriticalTemp',
    metricName = 'rmse'
)

print('Root Mean Square Error (rmse) on test data = %g' % myPredEvaluator.evaluate(predictions))

In [ ]:
myPredEvaluator = RegressionEvaluator (
    predictionCol = 'prediction',
    labelCol = 'CriticalTemp',
    metricName = 'r2'
)

print('R Squared (R2) on test data = %g' % myPredEvaluator.evaluate(predictions))

# Logistic Regression

In [ ]:
# split the dataset into 67% training and 33% test
(train, test) = df.randomSplit([0.67,0.33])

train.show(10, truncate = False)
test.show(10, truncate = False)

In [ ]:
numCols = [x for (x, dataType) in train.dtypes if ((dataType == 'double') & (x != 'isFound'))]
print(numCols)
assemblerInput = [x for x in numCols]
assemblerInput

In [ ]:
vectorAssembler = VectorAssembler (
    inputCols = assemblerInput,
    outputCol = 'VectorAssembler_features'
)
vectorAssembler

In [ ]:
stages = []
stages += [vectorAssembler]

stages

In [ ]:
from pyspark.ml import Pipeline

pipeline = Pipeline().setStages(stages)
model = pipeline.fit(train)

ppDf = model.transform(test)
ppDf.show(20)
ppDf.printSchema()

In [ ]:
# Import logistic Regression model
from pyspark.ml.classification import LogisticRegression
import pyspark.sql.functions as F
import pyspark.sql.types as T

In [ ]:
# # split the features and label. We want to determine the critical temperature, hence, CriticalTemp is the label.
# # exclude it from other features.
data = ppDf.select (
    F.col('vectorAssembler_features').alias('features'),
    F.col('CriticalTemp').alias('label')
)

In [ ]:
data.show(200)
data.printSchema()

In [ ]:
(Train, Test) = data.randomSplit([0.7,0.3], seed=42)

In [ ]:
model = LogisticRegression(labelCol='label')
lModel = model.fit(Train)
lModelSummary = lModel.summary

In [ ]:
lModelSummary.predictions.show()

In [ ]:
lModelSummary.predictions.describe().show()

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [ ]:
predCriticalTemp = lModel.evaluate(Test)
predCriticalTemp.predictions.show()

In [ ]:
eval = BinaryClassificationEvaluator(rawPredictionCol='prediction', labelCol='label')

In [ ]:
auc = eval.evaluate(predCriticalTemp.predictions)
auc